# GPUBMA — exhaustive 2^30 enumeration on an NVIDIA A100 (Google Colab)

Runs the **validated** GPUBMA exhaustive enumerator (streamed direct batching,
float64 only, deterministic reductions, checkpoint/resume — see
`docs/ADR_0001_GPU_ENUMERATOR.md` and `docs/FWL_BLOCK_FORMULATION.md`) on the
frozen `panel_30` dataset: **p = 30 optional predictors, 2^30 = 1,073,741,824
models**, shrink (Stata-verified) convention, controls `w1 w2`, g = 1000,
beta-binomial(1, 1) model prior.

**Before running:**
1. Copy the `gpubma` repository (including `data/synthetic/panel_30.parquet`
   and `panel_30_metadata.json`) to Google Drive at `MyDrive/GPUBMA`
   (or edit `REPO_DIR` below).
2. Runtime → Change runtime type → **A100 GPU**.

Checkpoints are written to Drive every 60 s, so a Colab disconnect loses at
most ~1 minute of work: **just reopen the notebook and Run all** — it
resumes from the checkpoint automatically, and if the run already finished
it loads the saved results instead of recomputing (idempotent).


In [ ]:
# [COLAB] Mount Google Drive and define all paths (POSIX only)
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

REPO_DIR = Path('/content/drive/MyDrive/GPUBMA')   # repository copy on Drive
WORK_DIR = Path('/content/drive/MyDrive/GPUBMA_p30_run')  # checkpoints + results
WORK_DIR.mkdir(parents=True, exist_ok=True)
assert (REPO_DIR / 'pyproject.toml').exists(), (
    f'gpubma repository not found at {REPO_DIR}; copy it to Drive first')
print('repo :', REPO_DIR)
print('work :', WORK_DIR)


In [ ]:
# [COLAB] Install gpubma from the repository copy (plus psutil for host memory)
%pip install --quiet $REPO_DIR/. psutil

import importlib.metadata, subprocess
GPUBMA_VERSION = importlib.metadata.version('gpubma')
try:
    GIT_COMMIT = subprocess.run(
        ['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'],
        capture_output=True, text=True, check=True).stdout.strip()
except Exception:
    GIT_COMMIT = 'unknown (repository copied without .git)'
print('gpubma', GPUBMA_VERSION, '@', GIT_COMMIT)


In [ ]:
# [COLAB] Verify CUDA, GPU name, VRAM, compute capability, and real float64
import torch

assert torch.cuda.is_available(), 'CUDA is not available — select a GPU runtime'
props = torch.cuda.get_device_properties(0)
free_b, total_b = torch.cuda.mem_get_info()
GPU_NAME = props.name
print(f'GPU: {GPU_NAME}  CC {props.major}.{props.minor}')
print(f'VRAM: total {total_b / 2**30:.1f} GiB, free {free_b / 2**30:.1f} GiB')
print(f'torch {torch.__version__} (CUDA {torch.version.cuda})')

# genuine float64 execution check (not just a capability flag)
a = torch.randn(256, 256, dtype=torch.float64, device='cuda')
r = (a @ a.T).sum()
torch.cuda.synchronize()
assert r.dtype == torch.float64
ref = (a.cpu().numpy() @ a.cpu().numpy().T).sum()
assert abs(float(r) - float(ref)) < 1e-6, 'float64 GPU result inconsistent'
print('float64 GPU execution: OK')

# CuPy is NOT used by gpubma (backend is torch); reported for completeness
try:
    import cupy
    print('cupy present (informational, unused):', cupy.__version__)
except ImportError:
    print('cupy not installed (fine — gpubma uses torch)')

ALLOW_NON_A100 = False  # set True only if you deliberately use another GPU
if 'A100' not in GPU_NAME and not ALLOW_NON_A100:
    raise RuntimeError(f'expected an A100, got {GPU_NAME}; '
                       'set ALLOW_NON_A100 = True to override')


In [ ]:
# [COLAB] gpubma doctor (hardware diagnostic, saved next to the results)
!python -m gpubma.doctor --json "$WORK_DIR/gpu_doctor_colab.json"


In [ ]:
# [COLAB] Run configuration (validated statistical setup; conservative VRAM)
P = 30                      # 2^30 = 1,073,741,824 models
TOP_K = 20
CHECKPOINT_EVERY_S = 60.0   # Drive checkpoint cadence (disconnect-safe)
PROGRESS_EVERY_S = 30.0

DATA_PARQUET = REPO_DIR / 'data' / 'synthetic' / 'panel_30.parquet'
META_JSON = REPO_DIR / 'data' / 'synthetic' / 'panel_30_metadata.json'
CKPT_PATH = WORK_DIR / f'enum_p{P}.ckpt.npz'
OUT_DIR = WORK_DIR / f'results_p{P}'
OUT_DIR.mkdir(exist_ok=True)

# conservative automatic batch sizing from ACTUAL free VRAM (A100 40 GB and
# 80 GB differ): ~25% of free memory as working budget, chunk cap scaled.
free_b, total_b = torch.cuda.mem_get_info()
VRAM_BUDGET = int(0.25 * free_b)
MAX_CHUNK = 1 << 17 if free_b > 30 * 2**30 else 1 << 16
print(f'VRAM budget {VRAM_BUDGET / 2**30:.1f} GiB, max chunk {MAX_CHUNK:,}')


In [ ]:
# [CORE] Load frozen panel_30, verify checksum, prepare validated inputs
import hashlib, json
import numpy as np
import pandas as pd

meta = json.loads(Path(META_JSON).read_text())
sha = hashlib.sha256(Path(DATA_PARQUET).read_bytes()).hexdigest()
assert sha == meta['file_checksums']['parquet']['sha256'], (
    'panel_30.parquet checksum mismatch — dataset is not the frozen artifact')
print('panel_30.parquet sha256 verified:', sha[:16], '...')

df = pd.read_parquet(DATA_PARQUET)
n = len(df)
assert n == 1000 and meta['expected_number_of_models'] == 1 << 30
predictors = [f'x{j}' for j in range(1, P + 1)]

# shrink (Stata-verified) convention: residualize on [1, w1, w2],
# df = n - 1, TSS_c normalizer, joint g-prior over optional + always slopes
y = df['y'].to_numpy(np.float64)
X = df[predictors].to_numpy(np.float64)
A = np.column_stack([np.ones(n), df[['w1', 'w2']].to_numpy(np.float64)])
Q, _ = np.linalg.qr(A)
y_r = y - Q @ (Q.T @ y)
X_r = X - Q @ (Q.T @ X)
yc = y - y.mean()
CONV = dict(df_resid=n - 1, tss_norm=float(yc @ yc), k_always=2)
G = float(max(n, P * P))  # = 1000 for n = 1000, P <= 31 — validated default

from gpubma.priors.model_priors import log_model_prior_function
LOG_PRIOR, prior_desc = log_model_prior_function(('betabinomial', 1.0, 1.0), P)
N_EXPECTED = 1 << P
print(f'p = {P}, models = {N_EXPECTED:,}, g = {G:.0f}, prior = {prior_desc}')


In [ ]:
# [CORE] Enumerate — resumes from Drive checkpoint; skips if already complete
import time
import gpubma.gpu.enumerator as _en
from gpubma.gpu.enumerator import enumerate_models_gpu

RESULTS_JSON = OUT_DIR / 'results.json'
result = None
if RESULTS_JSON.exists():
    print('final results already on Drive — skipping enumeration (idempotent)')
else:
    # count checkpoint writes for the benchmark report
    _orig_save = _en._save_checkpoint
    CKPT_WRITES = [0]
    def _counting_save(path, state):
        _orig_save(path, state)
        CKPT_WRITES[0] += 1
    _en._save_checkpoint = _counting_save

    t_wall = time.time()
    def show_progress(info):
        rate = info['models_per_second']
        remaining = (info['models_total'] - info['models_done']) / max(rate, 1e-9)
        print(f"[{info['fraction']:7.2%}] "
              f"{info['models_done']:,}/{info['models_total']:,} models  "
              f"k={info['current_size']:2d}  elapsed {info['elapsed_s']:,.0f} s  "
              f"{rate:,.0f} models/s  ETA {remaining:,.0f} s  "
              f"GPU {torch.cuda.memory_allocated() / 2**30:.2f} GiB "
              f"(peak {torch.cuda.max_memory_allocated() / 2**30:.2f} GiB)",
              flush=True)

    resume = CKPT_PATH.exists()
    print('resuming from Drive checkpoint' if resume else 'starting fresh')
    result = enumerate_models_gpu(
        X_r, y_r, g=G, log_model_prior=LOG_PRIOR, top_k=TOP_K,
        vram_budget_bytes=VRAM_BUDGET, max_chunk=MAX_CHUNK,
        checkpoint_path=CKPT_PATH, checkpoint_every_s=CHECKPOINT_EVERY_S,
        resume=resume, progress_every_s=PROGRESS_EVERY_S,
        progress=show_progress, **CONV)
    WALL_S = time.time() - t_wall
    N_CKPT = CKPT_WRITES[0]
    _en._save_checkpoint = _orig_save
    print(f"done: {result['n_models_evaluated']:,} models in "
          f"{result['runtime']['elapsed_s']:,.1f} s "
          f"({result['runtime']['models_per_second']:,.0f} models/s)")


In [ ]:
# [CORE] Exact-count assertion and posterior validation
if result is not None:
    assert result['n_models_evaluated'] == N_EXPECTED == (1 << P), (
        f"processed {result['n_models_evaluated']:,} != expected {N_EXPECTED:,}")
    if P == 30:
        assert result['n_models_evaluated'] == 1_073_741_824
    sd_sum = result['normalization_check']['size_distribution_sum']
    assert abs(sd_sum - 1.0) < 1e-9, f'size distribution sums to {sd_sum!r}'
    assert result['normalization_check']['pip_max_overshoot'] <= 1e-12
    assert (result['pip'] >= 0.0).all() and (result['pip'] <= 1.0).all()
    assert np.isfinite(result['log_normalizer'])
    assert result['runtime']['precision'] == 'float64'
    ms = float(np.arange(P + 1) @ result['size_distribution'])
    assert abs(ms - result['mean_model_size']) < 1e-9
    print(f"VALIDATED: exactly {result['n_models_evaluated']:,} models; "
          f"size-dist sum = {sd_sum:.15f}; PIPs in [0, 1]; "
          f"mean model size = {result['mean_model_size']:.6f}")
else:
    print('nothing to validate here — results were loaded below')


In [ ]:
# [CORE] Save results (JSON + CSV + Parquet) and the benchmark report
import platform, resource

def _results_payload(r):
    return {
        'p': P, 'g': G, 'model_prior': 'betabinomial(1,1)',
        'convention': 'shrink (Stata-verified)', 'precision': 'float64',
        'n_models_expected': int(N_EXPECTED),
        'n_models_evaluated': int(r['n_models_evaluated']),
        'log_normalizer': r['log_normalizer'],
        'mean_model_size': r['mean_model_size'],
        'pip': r['pip'].tolist(),
        'coef_mean': r['coef_mean'].tolist(),
        'coef_sd': r['coef_sd'].tolist(),
        'size_distribution': r['size_distribution'].tolist(),
        'top_models': r['top_models'],
        'normalization_check': r['normalization_check'],
        'runtime': r['runtime'],
    }

if result is not None:
    RESULTS_JSON.write_text(json.dumps(_results_payload(result), indent=2))
    payload = json.loads(RESULTS_JSON.read_text())
else:
    payload = json.loads(RESULTS_JSON.read_text())
    print('loaded previously saved results from Drive')

pred_tbl = pd.DataFrame({
    'predictor': predictors,
    'pip': payload['pip'],
    'posterior_mean': payload['coef_mean'],
    'posterior_sd': payload['coef_sd'],
})
size_tbl = pd.DataFrame({'model_size': range(P + 1),
                         'posterior_probability': payload['size_distribution']})
top_tbl = pd.DataFrame(payload['top_models'])
for name, tbl in [('predictors', pred_tbl), ('size_distribution', size_tbl),
                  ('top_models', top_tbl)]:
    tbl.to_csv(OUT_DIR / f'{name}.csv', index=False)
    tbl.to_parquet(OUT_DIR / f'{name}.parquet', index=False)

if result is not None:
    bench = {
        'label': 'Measured',
        'gpu': GPU_NAME,
        'runtime_s': payload['runtime']['elapsed_s'],
        'wall_clock_this_session_s': WALL_S,
        'models_per_second': payload['runtime']['models_per_second'],
        'peak_gpu_memory_bytes': payload['runtime']['peak_gpu_memory_bytes'],
        'peak_host_memory_kb_ru_maxrss':
            resource.getrusage(resource.RUSAGE_SELF).ru_maxrss,
        'checkpoint_writes_this_session': N_CKPT,
        'chunks': payload['runtime']['chunks'],
        'resumed': payload['runtime']['resumed'],
        'gpubma_version': GPUBMA_VERSION,
        'git_commit': GIT_COMMIT,
        'python': platform.python_version(),
        'torch': torch.__version__,
        'n_models_evaluated': payload['n_models_evaluated'],
        'validation': {
            'exact_count_ok': payload['n_models_evaluated'] == N_EXPECTED,
            **payload['normalization_check'],
        },
    }
    (OUT_DIR / 'benchmark_report.json').write_text(json.dumps(bench, indent=2))
print('saved to', OUT_DIR)
sorted(p.name for p in OUT_DIR.iterdir())


In [ ]:
# [CORE] Compact results table (paste into STATUS.md)
rt = payload['runtime']
print('| p | models | elapsed s | models/s | peak GPU GiB | device | '
      'size-dist sum | exact count |')
print('|---|---|---|---|---|---|---|---|')
print(f"| {P} | {payload['n_models_evaluated']:,} "
      f"| {rt['elapsed_s']:,.1f} | {rt['models_per_second']:,.0f} "
      f"| {rt['peak_gpu_memory_bytes'] / 2**30:.2f} | {rt['device']} "
      f"| {payload['normalization_check']['size_distribution_sum']:.15f} "
      f"| {'yes' if payload['n_models_evaluated'] == N_EXPECTED else 'NO'} |")
print()
print('top 5 models (mask, size, pmp):')
for m in payload['top_models'][:5]:
    print(f"  {m['mask']:>10d}  size {m['size']:2d}  pmp {m['pmp']:.6f}")


Reopening after a disconnect: run all cells again — the run resumes from the
newest Drive checkpoint (at most ~60 s of work lost), and once `results.json`
exists the notebook only reloads and re-exports without recomputing.
